# Extracción de Características Temporales (Feature Engineering) para sEMG

Este cuadernillo se encarga de realizar la **extracción manual de características (Feature Engineering)** a partir de las ventanas temporales de señal sEMG monocanal de tu prototipo Myotensor.

Extraeremos **6 características fundamentales en el dominio del tiempo (Time Domain Features)** recomendadas en la literatura científica para clasificar gestos utilizando algoritmos tradicionales como **Support Vector Machine (SVM)** y **Random Forest (RF)**:

1. **MAV** (Mean Absolute Value)
2. **RMS** (Root Mean Square)
3. **WL** (Waveform Length)
4. **ZC** (Zero Crossings) — con umbral de ruido basado en `noise_std`
5. **SSC** (Slope Sign Changes) — con umbral de ruido basado en `noise_std`
6. **VAR** (Variance)

## Carga de Datos

In [6]:
import numpy as np
import pandas as pd
import os
import joblib

# Función puramente robusta para buscar y cargar el archivo .env
def load_env_variables():
    import os
    from pathlib import Path
    
    # 1. Buscar .env subiendo niveles desde el CWD actual
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    # 2. Leer e inyectar variables en os.environ
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()

✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


## Definición de Funciones (Time Domain Features)

In [7]:
# 1. Definición de Fórmulas Matemáticas y Funciones Vectorizadas en NumPy
import json
import glob
from pathlib import Path

try:
    # Buscar metadata dinámicamente en RAW_DATA_PROTO
    raw_dir = Path(os.environ["RAW_DATA_PROTO"])
    meta_files = list(raw_dir.glob("**/session_*_metadata.json"))
    if meta_files:
        with open(meta_files[0], 'r') as f:
            meta = json.load(f)
        NOISE_STD = meta.get('calibration', {}).get('normalized_noise_std', 0.03188)
    else:
        NOISE_STD = 0.03188
except Exception as e:
    print(f'⚠️ No se pudo cargar el ruido normalizado: {e}')
    NOISE_STD = 0.03188

print(f'⚡ Umbral dinámico de ruido cargado: {NOISE_STD:.5f}')

def mean_absolute_value(x):
    return np.mean(np.abs(x))

def root_mean_square(x):
    return np.sqrt(np.mean(np.square(x)))

def waveform_length(x):
    return np.sum(np.abs(np.diff(x)))

def zero_crossings(x, threshold=NOISE_STD):
    x = np.asarray(x).flatten()
    sign_changes = (x[:-1] * x[1:]) < 0
    above_threshold = np.abs(x[:-1] - x[1:]) > threshold
    return np.sum(sign_changes & above_threshold)

def slope_sign_changes(x, threshold=NOISE_STD):
    x = np.asarray(x).flatten()
    d1 = x[1:-1] - x[:-2]
    d2 = x[2:] - x[1:-1]
    slope_changes = (d1 * d2) < 0
    above_threshold = (np.abs(d1) > threshold) & (np.abs(d2) > threshold)
    return np.sum(slope_changes & above_threshold)

def variance(x):
    return np.var(x, ddof=0)

# 2. Carga de los Tensores de Señal y Reconstrucción del Dataset Original
base_path = os.environ["PROCESSED_TENSOR_PROTO"]
print("Cargando tensores procesados de Myotensor Proto...")
X_train = np.load(f'{base_path}/X_train.npy')
X_test  = np.load(f'{base_path}/X_test.npy')
y_train_cat = np.load(f'{base_path}/y_train.npy')
y_test_cat  = np.load(f'{base_path}/y_test.npy')

scaler_raw = joblib.load(f'{base_path}/std_scaler.bin')
W = X_train.shape[1]

X_train_raw = scaler_raw.inverse_transform(X_train.reshape(-1, 1)).reshape(X_train.shape[0], W)
X_test_raw  = scaler_raw.inverse_transform(X_test.reshape(-1, 1)).reshape(X_test.shape[0], W)

X_data = np.concatenate([X_train_raw, X_test_raw], axis=0)
y_data = np.argmax(np.concatenate([y_train_cat, y_test_cat], axis=0), axis=1)

print(f"✅ X_data reconstruido exitosamente: {X_data.shape} ventanas.")
print(f"✅ y_data reconstruido: {y_data.shape} con clases {np.unique(y_data)}")


⚡ Umbral dinámico de ruido cargado: 0.02173
Cargando tensores procesados de Myotensor Proto...
✅ X_data reconstruido exitosamente: (4522, 300) ventanas.
✅ y_data reconstruido: (4522,) con clases [0 1 2 3]


## Extracción de Características (Transformación del Tensor)

In [8]:
print("⚡ Iniciando extracción manual de características...")
num_ventanas = X_data.shape[0]
X_features = np.zeros((num_ventanas, 6))

# Umbral de ruido basado en la calibración del protoboard (noise_std = 4.71)
noise_threshold = NOISE_STD

for i in range(num_ventanas):
    window_signal = X_data[i]
    X_features[i, 0] = mean_absolute_value(window_signal)
    X_features[i, 1] = root_mean_square(window_signal)
    X_features[i, 2] = waveform_length(window_signal)
    X_features[i, 3] = zero_crossings(window_signal, threshold=noise_threshold)
    X_features[i, 4] = slope_sign_changes(window_signal, threshold=noise_threshold)
    X_features[i, 5] = variance(window_signal)

print(f"¡Extracción finalizada con éxito!")
print(f"Dimensión final del tensor de características X_features: {X_features.shape}")

# Visualizar las primeras filas en un DataFrame de Pandas para verificación estética y numérica
feature_names = ['MAV', 'RMS', 'WL', 'ZC', 'SSC', 'VAR']
df_preview = pd.DataFrame(X_features, columns=feature_names)
df_preview['Clase'] = y_data
print("\n--- Vista Previa de Características Extraídas ---")
print(df_preview.head())

⚡ Iniciando extracción manual de características...
¡Extracción finalizada con éxito!
Dimensión final del tensor de características X_features: (4522, 6)

--- Vista Previa de Características Extraídas ---
        MAV       RMS         WL    ZC    SSC       VAR  Clase
0  0.109036  0.129419  24.241561  58.0  133.0  0.006343      0
1  0.106253  0.125491  23.087630  54.0  131.0  0.005807      0
2  0.106546  0.126016  23.078721  48.0  130.0  0.005738      0
3  0.104684  0.124661  22.254713  45.0  126.0  0.005737      0
4  0.106467  0.126098  21.647082  39.0  126.0  0.005525      0


## División de Datos y Estandarización (Vital para SVM)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Dividiendo el conjunto de características en Train (80%) y Test (20%)...")

# Split estratificado por clases (stratify=y_data) para mantener el balance de gestos
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_features,
    y_data,
    test_size=0.2,
    random_state=42,
    stratify=y_data
)

print(f"   * X_train_f: {X_train_f.shape} | y_train_f: {y_train_f.shape}")
print(f"   * X_test_f:  {X_test_f.shape}  | y_test_f:  {y_test_f.shape}")

print("\n⚡ Ajustando StandardScaler de Scikit-Learn...")
scaler_features = StandardScaler()

# Ajustar (fit) únicamente en entrenamiento para evitar Data Leakage
scaler_features.fit(X_train_f)

# Transformar conjuntos de entrenamiento y prueba
X_train_scaled = scaler_features.transform(X_train_f)
X_test_scaled  = scaler_features.transform(X_test_f)

print(f"✅ Conjunto de Entrenamiento Escalado: {X_train_scaled.shape}")
print(f"✅ Conjunto de Prueba Escalado:       {X_test_scaled.shape}")

Dividiendo el conjunto de características en Train (80%) y Test (20%)...
   * X_train_f: (3617, 6) | y_train_f: (3617,)
   * X_test_f:  (905, 6)  | y_test_f:  (905,)

⚡ Ajustando StandardScaler de Scikit-Learn...
✅ Conjunto de Entrenamiento Escalado: (3617, 6)
✅ Conjunto de Prueba Escalado:       (905, 6)


## Guardado de los Vectores Finales

In [10]:
# Se crea el subdirectorio 'vector_classic' en datasets desde la ruta absoluta .env
output_dir = os.environ["PROCESSED_VECTOR_CLASSIC_PROTO"]
os.makedirs(output_dir, exist_ok=True)

print(f"💾 Guardando matrices resultantes en: {output_dir} ...")
np.save(os.path.join(output_dir, 'X_train_scaled.npy'), X_train_scaled)
np.save(os.path.join(output_dir, 'X_test_scaled.npy'), X_test_scaled)
np.save(os.path.join(output_dir, 'X_train_raw.npy'), X_train_f)
np.save(os.path.join(output_dir, 'X_test_raw.npy'), X_test_f)
np.save(os.path.join(output_dir, 'y_train.npy'), y_train_f)
np.save(os.path.join(output_dir, 'y_test.npy'), y_test_f)

# Guardar también el escalador de características
scaler_path = os.path.join(output_dir, 'features_scaler.bin')
joblib.dump(scaler_features, scaler_path)

print("\n🎉 ¡Todo listo y guardado con éxito! Matrices listas para entrenar SVM y Random Forest.")

💾 Guardando matrices resultantes en: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/datasets/processed/myotensor_proto/vector_classic ...

🎉 ¡Todo listo y guardado con éxito! Matrices listas para entrenar SVM y Random Forest.
